# Settings

In [1]:
options(scipen=999)
pacman::p_load(dplyr, stringr, data.table, tidyr, plyr,
               pheatmap, colorRamps, gridExtra, ggplot2, ggrepel, RColorBrewer,ComplexHeatmap,vioplot,ggforce)


## Directories

In [2]:
# Set working directories
base.dir =  "/nfs/lab/projects/CellCrosstalk/npod.pancreas/Final/"
assets.dir = paste(base.dir, "Assets/", sep = "") 
results.dir = paste(base.dir, "Results/corrected/", sep = "")
permutation.result.dir = paste(base.dir, "Results/corrected.permutation.test/", sep = "")


# Databases
cc.db = "/nfs/lab/projects/islet_multiomics_stress/analysis/CellChat/Assets/CellChatDB.human.txt"
Gaulton.db.dir = "/nfs/lab/Luca/Assets/LR.Database/GaultonDB_V2.txt"

permutations.test.tmp.dir = paste(permutation.result.dir, "TMP/", sep = "")
permutations.test.res.dir = paste(permutation.result.dir, "Results/", sep = "")

cc.db = "/nfs/lab/projects/islet_multiomics_stress/analysis/CellChat/Assets/CellChatDB.human.txt"

## Variables

In [3]:
# conditions
condition.ls = c("ND", "Aab", "T1D_early","T1D_late")

colors.conditions = c("ND" = "#85EF1B",
                      "Aab" = "#EFEF1B",
                      "T1D_early" = "#EF851B",
                      "T1D_late" = "#EF1B1B")

# Contrasts
Contrast_1 = c(condition.ls[2], condition.ls[3], condition.ls[4])
Contrast_2 = c(condition.ls[1], condition.ls[1], condition.ls[1])

# Celltypes and compartments
Endocrine = c('Alpha', 'Beta', 'Delta')
Exocrine = c('Acinar1_2_6','Acinar_3','Acinar_4','Acinar_5','Ductal','MUC5b_Ductal')
Immune = c('Tcells','Macrophage')
Endothelial = c('Endothelial')
Stromal = c('Activated_Stellate','Quiescent_Stellate')

cell.pop.order = c(Endocrine, Exocrine, Immune, Endothelial, Stromal)
rev.cell.pop.order = c(rev(Stromal), rev(Endothelial), rev(Immune), rev(Exocrine), rev(Endocrine), use.names = TRUE)

compartment = cell.pop.order
compartment = ifelse(compartment  %in%  Endocrine, "Endocrine", 
                 ifelse(compartment  %in%  Exocrine, "Exocrine",
                        ifelse(compartment  %in%  Immune, "Immune",
                               ifelse(compartment  %in%  Endothelial, "Endothelial", "Stromal"))))

Cellpop.compartment = as.data.frame(cbind(cell.pop.order, compartment))
Cellpop.compartment$compartment = factor(Cellpop.compartment$compartment, levels = c('Endocrine','Exocrine','Immune','Endothelial', "Stromal"))
colnames(Cellpop.compartment)[1] = "Celltype"

colors.compartments = c('Endocrine'='#26bfbf',
           'Exocrine'='#ed872d',
           'Immune'='#3F98E0',
           'Endothelial'='#800080',
           'Stromal'='#F5DE6C')

Cellpop.compartment$colors.compartments = compartment
Cellpop.compartment$colors.compartments = mapvalues(Cellpop.compartment$colors.compartments, names(colors.compartments), colors.compartments, warn_missing = TRUE)
gaps.compartments = c(3, 9, 11, 12)

# Pre-Processing

In [5]:
setwd(permutation.result.dir)
# Set number of permutations
nsim = 100
# Load variables
LR.list <- dir(pattern = "_trim0.txt")
# Use same threshold used for the observations
sig.tresh = 0.1

In [7]:
for (i in 1:nsim){
    nboot = paste("nboot", i, "_trim0.txt", sep = "")
    LR.list.use = LR.list[grepl(nboot, LR.list)]
    message("Processing simulation #: " , i)
    # Reset consensus
    LR.data.sim.consensus = NULL
    for (c.i in seq_along(condition.ls)){
        setwd(permutation.result.dir)
        lr.list.condition.use = LR.list.use[grepl(condition.ls[c.i], LR.list.use)]
        message("Processing ", lr.list.condition.use)
        # Read data
        LR.data.sim <- read.table(lr.list.condition.use, sep = "\t", stringsAsFactors = F, header = T)
        message("\t Detected: ", nrow(LR.data.sim), " interactions")
        # Add interaction ID column
        LR.data.sim$interaction.id = paste(LR.data.sim$source, 
                                                            LR.data.sim$target, 
                                                            LR.data.sim$ligand,
                                                            LR.data.sim$receptor,
                                                            sep = "--")
        # Pull all conditions together
        Prefix <- strsplit(lr.list.condition.use, "_df.net")[[1]][1]
        LR.data.sim$condition = Prefix
        LR.data.sim.consensus <- rbind(LR.data.sim.consensus, LR.data.sim)
    }
    # Filter in the same way we filtered the observations         
    # FDR correction
    LR.data.sim.consensus$pval[LR.data.sim.consensus$pval == 0] = 0.0001 # replacing 0s with a low number
    LR.data.sim.consensus$FDR = p.adjust(LR.data.sim.consensus$pval, 'BH')
    LR.data.sim.consensus = LR.data.sim.consensus[LR.data.sim.consensus$FDR < sig.tresh, ]
    # IS tresholding
    quantiles = as.vector(quantile(LR.data.sim.consensus$prob))
    IS.tresh = quantiles[2]
    LR.data.sim.consensus = LR.data.sim.consensus[LR.data.sim.consensus$prob > IS.tresh, ]
    print(paste0("Final # of LR pairs: ", nrow(LR.data.sim.consensus), 
        " - FDR theshold used: ", sig.tresh,
        " - IS theshold used: ", IS.tresh))
    
    # Blacklisting
    Blacklist.genes = c("INS")
    Whitelist.pops = c("Beta")
    Blacklist.pops = cell.pop.order[!cell.pop.order %in% Whitelist.pops]
    message("Removing ", Blacklist.genes, " from:")
    print(Blacklist.pops)
    LR.data.sim.consensus = LR.data.sim.consensus[!c(LR.data.sim.consensus$source %in% Blacklist.pops & LR.data.sim.consensus$ligand %in% Blacklist.genes),]

    Blacklist.genes = c("GCG")
    Whitelist.pops = c("Alpha")
    Blacklist.pops = cell.pop.order[!cell.pop.order %in% Whitelist.pops]
    message("Removing ", Blacklist.genes, " from:")
    print(Blacklist.pops)
    LR.data.sim.consensus = LR.data.sim.consensus[!c(LR.data.sim.consensus$source %in% Blacklist.pops & LR.data.sim.consensus$ligand %in% Blacklist.genes),]

    Blacklist.genes = c("SST")
    Whitelist.pops = c("Delta")
    Blacklist.pops = cell.pop.order[!cell.pop.order %in% Whitelist.pops]
    message("Removing ", Blacklist.genes, " from:")
    print(Blacklist.pops)
    LR.data.sim.consensus = LR.data.sim.consensus[!c(LR.data.sim.consensus$source %in% Blacklist.pops & LR.data.sim.consensus$ligand %in% Blacklist.genes),]

    Blacklist.genes = c("PRSS1", "PRSS2", "PRSS3")
    Whitelist.pops = c("Acinar1_2_6", "Acinar_3", "Acinar_4", "Acinar_5")
    Blacklist.pops = cell.pop.order[!cell.pop.order %in% Whitelist.pops]
    message("Removing ", Blacklist.genes, " from:")
    print(Blacklist.pops)
    LR.data.sim.consensus = LR.data.sim.consensus[!c(LR.data.sim.consensus$source %in% Blacklist.pops & LR.data.sim.consensus$ligand %in% Blacklist.genes),]

    # Receptors
    Blacklist.genes = c("CD8A", "CD8B", "CD8B2")
    Whitelist.pops = c("Tcells")
    Blacklist.pops = cell.pop.order[!cell.pop.order %in% Whitelist.pops]
    message("Removing ", Blacklist.genes, " from:")
    print(Blacklist.pops)
    LR.data.sim.consensus = LR.data.sim.consensus[!c(LR.data.sim.consensus$target %in% Blacklist.pops & LR.data.sim.consensus$receptor %in% Blacklist.genes),]

    Blacklist.genes = c("CD4")
    Whitelist.pops = c("Tcells", "Macrophage")
    Blacklist.pops = cell.pop.order[!cell.pop.order %in% Whitelist.pops]
    message("Removing ", Blacklist.genes, " from:")
    print(Blacklist.pops)
    LR.data.sim.consensus = LR.data.sim.consensus[!c(LR.data.sim.consensus$target %in% Blacklist.pops & LR.data.sim.consensus$receptor %in% Blacklist.genes),]
    print(paste0("Final # of LR pairs: ", nrow(LR.data.sim.consensus)))
    
    # Save dataframe
    setwd(permutations.test.tmp.dir)
    file.name = paste("CellChat_Permutation_n", i, "_DFnet.txt", sep = "")
    write.table(LR.data.sim.consensus, file.name, sep = "\t", quote = F, col.names=T, row.names = F)
}

Processing simulation #: 1

Processing ND_df.net_nboot1_trim0.txt

	 Detected: 39619 interactions

Processing Aab_df.net_nboot1_trim0.txt

	 Detected: 51510 interactions

Processing T1D_early_df.net_nboot1_trim0.txt

	 Detected: 44812 interactions

Processing T1D_late_df.net_nboot1_trim0.txt

	 Detected: 41266 interactions



[1] "Final # of LR pairs: 100609 - FDR theshold used: 0.1 - IS theshold used: 0.0000190067034354562"


Removing INS from:



 [1] "Alpha"              "Delta"              "Acinar1_2_6"       
 [4] "Acinar_3"           "Acinar_4"           "Acinar_5"          
 [7] "Ductal"             "MUC5b_Ductal"       "Tcells"            
[10] "Macrophage"         "Endothelial"        "Activated_Stellate"
[13] "Quiescent_Stellate"


Removing GCG from:



...

 [1] "Alpha"              "Beta"               "Delta"             
 [4] "Ductal"             "MUC5b_Ductal"       "Tcells"            
 [7] "Macrophage"         "Endothelial"        "Activated_Stellate"
[10] "Quiescent_Stellate"


Removing CD8ACD8BCD8B2 from:



 [1] "Alpha"              "Beta"               "Delta"             
 [4] "Acinar1_2_6"        "Acinar_3"           "Acinar_4"          
 [7] "Acinar_5"           "Ductal"             "MUC5b_Ductal"      
[10] "Macrophage"         "Endothelial"        "Activated_Stellate"
[13] "Quiescent_Stellate"


Removing CD4 from:



 [1] "Alpha"              "Beta"               "Delta"             
 [4] "Acinar1_2_6"        "Acinar_3"           "Acinar_4"          
 [7] "Acinar_5"           "Ductal"             "MUC5b_Ductal"      
[10] "Endothelial"        "Activated_Stellate" "Quiescent_Stellate"
[1] "Final # of LR pairs: 100445"


In [8]:
# What to check
## 1) Check that different simulations (same condition) have different numbers
## 2) Check that all simulations/condition always have different numbers
## 3) Check that all simulations have different IS tresh